In [1]:

from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

set_llm_cache(SQLiteCache(database_path="cache/langchain.db"))

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

token_provider = get_bearer_token_provider(  
            DefaultAzureCredential(),  
            "https://cognitiveservices.azure.com/.default"  
        )

In [12]:
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI

llm = AzureChatOpenAI(
    deployment_name="gpt-4o",
    azure_ad_token_provider = token_provider,
    temperature = 0
    )

embeddings = AzureOpenAIEmbeddings(
    azure_deployment = 'text-embedding-ada-002',
    azure_ad_token_provider = token_provider
)

In [3]:
import pandas as pd

pitti = pd.read_excel('data_excel/pitti.xlsx')
pitti

,description,tag,link
0,The company was founded in 1972 when Goliardo ...,"leather wear, casual jackets, artisan attentio...",/en/e-pitti/fairs/pittiuomo97/exhibitors/A/a-f...
1,A. Leyva S.A. was founded in the early 1960's ...,"Play video, leather wear, braces, belts, monk,...",/en/e-pitti/fairs/pittiuomo97/exhibitors/A/a--...
2,A.B.C.L. born by passion and vision shared by ...,"shirts, casual jackets, denim, trousers, t-shi...",/en/e-pitti/fairs/pittiuomo97/exhibitors/A/a-b...
3,The purity of the forms and the perfect harmon...,"leather wear, coats, jackets, casual jackets, ...",/en/e-pitti/fairs/pittiuomo97/exhibitors/A/a-g...
4,A.S.98 was established in 2014 as a rebranding...,"Play video, braces, belts, wallets/small leath...",/en/e-pitti/fairs/pittiuomo97/exhibitors/A/a-s-98
...,...,...,...
792,Handcrafted scarves Made ...,"Play video, scarves, scarf, artisan attention,...",/en/e-pitti/fairs/pittiuomo97/exhibitors/0-9/1...
793,24Bottles is the Italian design brand born in ...,"shelf-items, agender, comfort, contemporary me...",/en/e-pitti/fairs/pittiuomo97/exhibitors/0-9/2...
794,Quality Originality Care ...,"Play video, sneakers, urban attitude, denimwea...",/en/e-pitti/fairs/pittiuomo97/exhibitors/0-9/2...
795,40Weft is a new concept of fashion wear. Betwe...,"shirts, denim, casual jackets, knitwear, trous...",/en/e-pitti/fairs/pittiuomo97/exhibitors/0-9/4...


In [5]:
pitti_desc = pitti['description']

In [16]:
from langchain_core.documents import Document

documents = []

for desc in pitti_desc:
    doc = Document(
        page_content=str(desc)
        )
    documents.append(doc)

documents = documents[:50]

In [17]:
documents

[Document(metadata={}, page_content='The company was founded in 1972 when Goliardo and Franco Lazzeri together managed to combine the Italian style to the tradition and to the craft passion. The company was initially specialized exclusively in the production of shearling and Lapin (anagram of the name Nipal). The secret of NIPAL success is guaranteed by the combination of experienced craftsmen’ s skills and the passion and believe of the management. Sartorial know-how, a high quality product, the craftsmanship secrets of the Made in Italy and the creative impulse of the company to always look at the future, have always been Nipal’s fundamental characteristics. Nipal is keeper of the brands Fontanelli, AFG 1972, Stilnology and Reali26. Nipal designs and makes leather and fabric jackets for man and woman. Creativity, imagination and personality always looking for new trends as well as the ability to anticipate patterns and to design ideas with large intuition are the key success of the c

In [18]:
from langchain_community.vectorstores.faiss import FAISS

vs = FAISS.from_documents(documents, embeddings)

In [24]:
query = 'what happened in london'
retr_docs = vs.similarity_search(query, 5)
retr_docs

[Document(id='fd76a20d-d41c-4cf8-8775-92e91497a5c3', metadata={}, page_content='Founded in London in 2004, Amplified became quickly established as the leading music-inspired lifestyle brand, specialising in artwork for retro and classic rock bands. Amplified Clothing went on to dominate the market.Musicians, celebrities, music fans and fashion enthusiasts across the world adopted Amplified as a firm favourite. Over a decade later and the brand continues to explore a wider world of music from genres and sub-genres that have never stopped influencing our lives. Music is the absolute inspiration for everything we do. Amplified makes some noise.'),
 Document(id='ee1e29ec-9255-4e85-a376-8990a519a432', metadata={}, page_content='nan'),
 Document(id='f6a013d4-0a9b-4adf-b2ce-aaadcaec10be', metadata={}, page_content='nan'),
 Document(id='d52a0ba0-76bd-4c09-88eb-acee1a17eb27', metadata={}, page_content='A.B.C.L. born by passion and vision shared by Antonio and Mattia are the stitchings that bind

In [25]:
content = []
for doc in retr_docs:
    content.append(doc.page_content)


content = ' '.join(content)
content

'Founded in London in 2004, Amplified became quickly established as the leading music-inspired lifestyle brand, specialising in artwork for retro and classic rock bands. Amplified Clothing went on to dominate the market.Musicians, celebrities, music fans and fashion enthusiasts across the world adopted Amplified as a firm favourite. Over a decade later and the brand continues to explore a wider world of music from genres and sub-genres that have never stopped influencing our lives. Music is the absolute inspiration for everything we do. Amplified makes some noise. nan nan A.B.C.L. born by passion and vision shared by Antonio and Mattia are the stitchings that bind two different worlds and lifestyles into one brand. Part residing in Tokyo and the other in Venice, we have the desire to bring the best of what’s Japanese and seamlessly combine that with international design. A.B.C. represents the start of things, the beginning of a masterpiece, and the L. derives from Antonio surname. We a

In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

pt = ChatPromptTemplate.from_template("""
        Answer the user query: {query} given the context: {context}.  
        Do not create yourself the answer, just use the context provided to answer the user query.       
"""
    )

parser = StrOutputParser()
chain = pt | llm | parser

chain.invoke({
    'context': content, 
    'query': query
})

'The context provided does not contain specific information about events or occurrences in London. It mentions Amplified Clothing, a music-inspired lifestyle brand founded in London in 2004, and its influence in the fashion and music world. Additionally, it references Camden Market in London as a source of inspiration for the Anerkjendt brand. However, it does not detail any specific events or happenings in London.'

In [33]:
k = 5
def rag(query, k, vs):
    retr_docs = vs.similarity_search(query, k)

    content = []
    for doc in retr_docs:
        content.append(doc.page_content)
    content = ' '.join(content)

    pt = ChatPromptTemplate.from_template("""
        Answer the user query: {query} given the context: {context}.  
        Do not create yourself the answer, just use the context provided to answer the user query.       
    """
        )

    parser = StrOutputParser()
    chain = pt | llm | parser

    response = chain.invoke({
        'context': content, 
        'query': query
    })
    
    return response



In [34]:
rag('what happened in london', k, vs)

'The context provided does not contain specific information about events or occurrences in London. It mentions Amplified Clothing, a music-inspired lifestyle brand founded in London in 2004, and its influence in the fashion and music world. Additionally, it references Camden Market in London as a source of inspiration for the Anerkjendt brand. However, it does not detail any specific events or happenings in London.'

In [35]:
rag('tell me something about Made in Italy', k, vs)

'"Made in Italy" represents a symbol of excellence and quality in craftsmanship, particularly in the fashion and luxury goods sectors. The context highlights several Italian companies that embody this tradition. Antonio Maurizi and A.Testoni are renowned for their artisanal shoemaking, with a focus on innovation and design, using 100% Italian components sourced for quality and environmental impact. A.Testoni, established in Bologna, has grown into an international luxury brand known for its unique combination of heritage and modern design techniques. Similarly, Alea, rooted in Romagna, has been a leader in high-quality shirt production since the 1950s, maintaining a balance between tradition and modernity. Alberto del Biondi S.p.A., founded in Padua, is a multidisciplinary company that combines design, innovation, and engineering, creating footwear, bags, and accessories for top international brands. These companies exemplify the "Made in Italy" ethos through their commitment to qualit

In [37]:
print(rag('which brand use innovation', k, vs))

The brands that use innovation according to the context provided are:

1. A.Testoni - The company has grown into an international luxury brand through constant innovation and design research, particularly in the use of leather, materials, and design.

2. AlphaTauri - This brand offers innovative fashion technologies integrated with high-quality and uniquely designed styles, combining textile innovations, purposeful design, and premium materials.


In [38]:
print(rag('which brand is a family business', k, vs))

The brand that is a family business given the context is Nipal, founded by Goliardo and Franco Lazzeri.


In [40]:
print(rag('are there any brands based on Naples', k, vs))

Yes, based on the context provided, Nipal is a company based on Naples.


In [41]:
print(rag('are there any brands specialized in jackets?', k, vs))

Yes, there are brands specialized in jackets mentioned in the context. Nipal designs and makes leather and fabric jackets for men and women. Additionally, Arctic Explorer presents different pieces of high-quality winter outerwear, including down jackets.


In [42]:
print(rag('are there any brands US based?', k, vs))

Yes, Allen Edmonds is a U.S.-based brand.


In [43]:
print(rag('list all brands that create products based on artisanal spirit', k, vs))

Based on the context provided, the brands that create products based on an artisanal spirit are:

1. Andrea Greco
2. Alessandro Gherardi
3. Antonio Maurizi
4. Nipal (including the brands Fontanelli, AFG 1972, Stilnology, and Reali26)
5. A.B.C.L.


In [44]:
print(rag('which products alessandro gherardi create?', k, vs))

Alessandro Gherardi creates belts, leather jewelry, small leather goods, bags, placemats, pillows, and carpets.


In [46]:
print(rag('what is the must to have from alessandro gherardi', k, vs))

The must-have from Alessandro Gherardi, given the context, would be items from the Alex Ingh collection. This collection is 100% Made in Italy and features high-range innovative tailoring with the finest Italian fabrics and original printings with exclusive details. It appeals to a younger audience with a sense of classical and traditional themes, offering a wardrobe that interprets the needs and personality of its user.
